# Zbus Construction Algorithm

**Author:** Leonardo Gabriel Granda Erazo
**Semester:** Seven
**Course:** Electric Power Systems

**Reference:** Grainger, J. J. & Stevenson, W. D. Jr. — *Power System Analysis*
**Chapter 8:** The impedance model and network calculations

This notebook builds the bus impedance matrix (**Zbus**) one branch at a time,
starting from an empty system. Each time a new impedance `Zb` is added, one of
four standard modification cases is applied:

| Case | Branch added | Effect on Zbus |
|------|--------------|----------------|
| **1** | New bus `p` ↔ reference node | Adds a new row/column; only the new diagonal element holds `Zb`. |
| **2** | New bus `p` ↔ existing bus `k` | Adds a new row/column copied from bus `k`; new diagonal element is `Zkk + Zb`. |
| **3** | Existing bus `k` ↔ reference node | Creates a temporary bus, then removes it via **Kron reduction**. |
| **4** | Existing bus `j` ↔ existing bus `k` | Creates a temporary bus from the difference of rows/columns `j` and `k`, then removes it via **Kron reduction**. |

Cases 3 and 4 introduce a temporary (fictitious) bus that is afterwards eliminated
with the **Kron reduction**, so the final matrix keeps the correct dimension.


## How to use

Run the cells in order: first `import numpy`, then the algorithm cell. The
algorithm cell starts an interactive session that asks you, step by step, how to
grow the matrix:

1. **Start / continue** — answer `yes` to add a branch, or `no` to finish.
2. **Case** — choose `1`, `2`, `3` or `4` (see the table above).
3. **Impedance `Zb`** — type a complex number, e.g. `0.25j` or `0.1+0.25j`.
4. **Extra data**, depending on the case:
   - *Case 4* also asks for the two existing buses `j` and `k` (they must be different).
   - *Cases 3 and 4* ask for the **index order** of the Kron reduction: a
     permutation of `0..n-1`. Typing the indices in order (e.g. `0,1,2,3`) keeps
     the temporary bus last, which is the usual choice.

Every answer is validated, so a typo just repeats the question instead of
crashing. The current matrix `Zbus` is printed after each step, and you can
inspect it at any time once the loop ends.

> **Worked example (Grainger & Stevenson):** Case 1 `Zb=1.25j` → Case 2 `Zb=0.25j`
> on bus 1 → Case 2 `Zb=0.4j` on bus 2 → Case 3 `Zb=1.25j` → Case 2 `Zb=0.2j` →
> Case 4 `Zb=0.125j`, `j=2`, `k=4`. The final result is a 4×4 Zbus matrix.


In [1]:
import numpy as np

In [ ]:
# Number of buses currently in the system (grows as branches are added).
bus_count = 0

# The bus impedance matrix. It starts empty and is rebuilt on every step.
Zbus = np.array([], dtype=complex)


# ============================================================================
#  Input helpers
# ----------------------------------------------------------------------------
#  These small functions keep asking until the user provides a valid value, so
#  a typo simply repeats the question instead of crashing the program or
#  corrupting the matrix.
# ============================================================================
def ask_yes_no(prompt):
    """Ask a yes/no question and keep asking until the answer is understood.
    Accepts yes/y -> True and no/n -> False (case-insensitive)."""
    while True:
        answer = input(prompt).strip().lower()
        if answer in ('yes', 'y'):
            return True
        if answer in ('no', 'n'):
            return False
        print('  Please answer "yes" or "no".')


def ask_case():
    """Ask which modification case to apply; only 1, 2, 3 or 4 are accepted."""
    while True:
        case = input('Enter the case to apply (1, 2, 3, 4): ').strip()
        if case in ('1', '2', '3', '4'):
            return case
        print('  Invalid case. Please type 1, 2, 3 or 4.')


def ask_impedance(prompt='Enter the impedance Zb (e.g. 0.25j): '):
    """Ask for a complex impedance and keep asking until it can be parsed."""
    while True:
        try:
            return complex(input(prompt))
        except ValueError:
            print('  Invalid impedance. Use a value like 0.25j or 0.1+0.25j.')


def ask_bus(prompt, n_buses):
    """Ask for an existing bus number within the range 1..n_buses."""
    while True:
        try:
            bus = int(input(prompt))
        except ValueError:
            print('  Invalid bus. Please type a whole number.')
            continue
        if 1 <= bus <= n_buses:
            return bus
        print(f'  Bus out of range. Choose a value between 1 and {n_buses}.')


def ask_order(n):
    """Ask for the index order used by the Kron reduction.
    It must be a permutation of 0..n-1 (typing them in order keeps the
    temporary bus in the last position, which is the usual choice)."""
    while True:
        text = input(f'Enter the new index order, comma separated '
                     f'(a permutation of 0..{n-1}): ')
        try:
            order = list(map(int, text.split(',')))
        except ValueError:
            print('  Invalid order. Type integers separated by commas.')
            continue
        if sorted(order) == list(range(n)):
            return order
        print(f'  Invalid order. It must be a permutation of 0..{n-1}.')


# ============================================================================
#  Kron reduction
# ----------------------------------------------------------------------------
#  Eliminates the temporary (fictitious) bus used in cases 3 and 4.
#
#  The matrix is first reordered so the bus to be removed sits in the last
#  row and column. It is then seen as a partitioned matrix:
#
#         | Zaa  Zab |
#         | Zba  Zbb |
#
#  and the reduced matrix is obtained with the Kron formula:
#
#         Z_reduced = Zaa - Zab * inv(Zbb) * Zba
#
#  Because the Zbus matrix is symmetric, Zba is just the transpose of Zab.
#
#  Parameters
#  ----------
#  Z_with_temp : matrix that still contains the temporary bus.
#  n           : matrix dimension (number of buses, temporary one included).
#  order       : index order used to move the temporary bus to the last
#                position before partitioning.
# ============================================================================
def kron_reduction(Z_with_temp, n, order):
    # Reorder columns and then rows so the bus to eliminate ends up last.
    Z_cols_reordered = Z_with_temp[:, order]
    Z_reordered = Z_cols_reordered[order, :]

    # Sub-blocks of the partitioned matrix:
    Zaa = np.empty((n - 1, n - 1), dtype=complex)  # buses we keep
    Zbb = np.empty((1, 1), dtype=complex)          # bus to remove
    Zab = np.empty((n - 1, 1), dtype=complex)      # coupling block

    # Split the reordered matrix into the three blocks above.
    for r in range(n):
        for c in range(n):
            if (r < n - 1) and (c < n - 1):
                Zaa[r, c] = Z_reordered[r, c]
            elif (r < n - 1) and (c == n - 1):
                Zab[r, 0] = Z_reordered[r, c]
            elif (r == n - 1) and (c == n - 1):
                Zbb[0, 0] = Z_reordered[r, c]

    # Apply the Kron reduction formula and return the smaller matrix.
    Z_reduced = Zaa - (Zab * np.linalg.inv(Zbb) * np.transpose(Zab))
    return Z_reduced


# ============================================================================
#  Main interactive loop
# ----------------------------------------------------------------------------
#  Repeatedly asks the user whether to add a branch and which case to apply,
#  rebuilding Zbus each time. Answer "no" to stop.
# ============================================================================
while True:
    # Ask to start the matrix (empty system) or to add another branch.
    if bus_count <= 0:
        keep_going = ask_yes_no('Do you want to start the Zbus matrix? (yes/no): ')
    else:
        keep_going = ask_yes_no('Do you want to add another bus? (yes/no): ')

    # User does not want to continue: exit the loop.
    if not keep_going:
        break

    print()
    case = ask_case()
    # Each case provisionally adds one bus (cases 3 and 4 undo this later
    # because their temporary bus is removed by the Kron reduction).
    bus_count += 1

    # ----------------------------------------------------------------
    #  CASE 1: new bus connected to the reference node.
    #  The new row/column is all zeros except the new diagonal = Zb.
    # ----------------------------------------------------------------
    if case == '1':
        print()
        print('Case 1: add Zb between a new bus and the reference node')
        print('Current buses: ', bus_count)
        Zb = ask_impedance()
        Z_new = np.zeros((bus_count, bus_count), dtype=complex)
        if bus_count <= 1:
            # Very first bus: a 1x1 matrix whose only element is Zb.
            Z_new[bus_count-1, bus_count-1] = Zb
        else:
            for row in range(bus_count):
                for col in range(bus_count):
                    if (row < bus_count-1) and (col < bus_count-1):
                        # Copy the existing matrix into the top-left block.
                        Z_new[row, col] = Zbus[row, col]
                    elif (row, col) == (bus_count-1, bus_count-1):
                        # New diagonal element holds Zb.
                        Z_new[row, col] = Zb
                    else:
                        # New row/column coupling with the rest is zero.
                        Z_new[row, col] = 0

    # ----------------------------------------------------------------
    #  CASE 2: new bus connected to an existing bus k.
    #  The new row/column copies bus k; new diagonal = Zkk + Zb.
    # ----------------------------------------------------------------
    elif case == '2':
        print()
        print('Case 2: add Zb between an existing bus and a new bus')
        print('Current buses: ', bus_count)
        Zb = ask_impedance()
        Z_new = np.zeros((bus_count, bus_count), dtype=complex)
        for row in range(bus_count):
            for col in range(bus_count):
                if (row < bus_count-1) and (col < bus_count-1):
                    # Keep the existing matrix.
                    Z_new[row, col] = Zbus[row, col]
                elif (row == bus_count-1) and (col < bus_count-1):  # Copy the row
                    Z_new[row, col] = Zbus[row-1, col]
                elif (row < bus_count-1) and (col == bus_count-1):  # Copy the column
                    Z_new[row, col] = Zbus[row, col-1]
                elif (row, col) == (bus_count-1, bus_count-1):      # New diagonal Zpp
                    Z_new[row, col] = Zb + Zbus[row-1, col-1]

    # ----------------------------------------------------------------
    #  CASE 3: existing bus connected to the reference node.
    #  Build a temporary bus like case 2, then remove it via Kron.
    # ----------------------------------------------------------------
    elif case == '3':
        print()
        print('Case 3: add Zb between an existing bus and the reference node through a temporary bus')
        print('Current buses: ', bus_count)
        Zb = ask_impedance()
        Z_new = np.zeros((bus_count, bus_count), dtype=complex)
        for row in range(bus_count):
            for col in range(bus_count):
                if (row < bus_count-1) and (col < bus_count-1):
                    # Keep the existing matrix.
                    Z_new[row, col] = Zbus[row, col]
                elif (row == bus_count-1) and (col < bus_count-1):  # Copy the row
                    Z_new[row, col] = Zbus[row-1, col]
                elif (row < bus_count-1) and (col == bus_count-1):  # Copy the column
                    Z_new[row, col] = Zbus[row, col-1]
                elif (row, col) == (bus_count-1, bus_count-1):      # Temporary diagonal Zpp
                    Z_new[row, col] = Zb + Zbus[row-1, col-1]

        # Eliminate the temporary bus and shrink the matrix back.
        print('------------ KRON REDUCTION ------------')
        order = ask_order(bus_count)
        Z_new = kron_reduction(Z_new, bus_count, order)
        print('The temporary bus is removed with the Kron method.')
        bus_count -= 1

    # ----------------------------------------------------------------
    #  CASE 4: branch between two existing buses j and k.
    #  Build a temporary bus from the difference of rows/columns j and k,
    #  then remove it via Kron.
    # ----------------------------------------------------------------
    elif case == '4':
        print()
        print('Case 4: add Zb between two existing buses through a temporary bus')
        print('Current buses: ', bus_count)
        Zb = ask_impedance()
        Z_new = np.zeros((bus_count, bus_count), dtype=complex)
        # j and k must be existing buses (the temporary bus is not counted yet).
        j = ask_bus('Enter bus j: ', bus_count - 1)
        while True:
            k = ask_bus('Enter bus k: ', bus_count - 1)
            if k != j:
                break
            print('  Bus k must be different from bus j.')
        for row in range(bus_count):
            for col in range(bus_count):
                if (row < bus_count-1) and (col < bus_count-1):
                    # Keep the existing matrix.
                    Z_new[row, col] = Zbus[row, col]
                elif (row == bus_count-1) and (col < bus_count-1):  # Row j minus row k
                    Z_new[row, col] = Zbus[j-1, col] - Zbus[k-1, col]
                elif (row < bus_count-1) and (col == bus_count-1):  # Column j minus column k
                    Z_new[row, col] = Zbus[row, j-1] - Zbus[row, k-1]
                elif (row, col) == (bus_count-1, bus_count-1):      # Temporary diagonal Zqq
                    Z_new[row, col] = Zb + (Zbus[j-1, j-1] + Zbus[k-1, k-1] - 2*Zbus[j-1, k-1])
        print(Z_new)

        # Eliminate the temporary bus and shrink the matrix back.
        print('------------ KRON REDUCTION ------------')
        order = ask_order(bus_count)
        Z_new = kron_reduction(Z_new, bus_count, order)
        print('The temporary bus is removed with the Kron method.')
        bus_count -= 1

    # Commit the new matrix and display it.
    Zbus = Z_new
    print(Zbus)
